In [3]:
import pandas as pd
import liana as li
liana_df = pd.read_parquet("/Users/k23030440/epitome_code/epitome/data/lig_rec/v_0.02/liana_consensus.parquet")
#from gene replace _cells with ""
# 
# Process the data
liana_df["ligand_complex"] = liana_df["gene"].str.split("___").str[0]
liana_df["receptor_complex"] = liana_df["gene"].str.split("___").str[1]

liana_df["target"] = liana_df["gene"].str.split("___").str[2]
liana_df["source"] = liana_df["gene"].str.split("___").str[3]

# remove those interactions that exist twice, both as source and target and target and source.
# Create a sorted interaction pair column to identify duplicates
liana_df["interaction"] = liana_df.apply(
    lambda row: tuple(
        sorted(
            [
                row["ligand_complex"],
                row["receptor_complex"],
                row["source"],
                row["target"],
            ]
        )
    ),
    axis=1,
)
# sort by ligand_complex
liana_df = liana_df.sort_values(
    ["ligand_complex", "corrected_score_y"], ascending=[True, True]
)
# Drop duplicate interactions
liana_df = liana_df.drop_duplicates(subset=["interaction"], keep="first")

# Drop the helper column
liana_df = liana_df.drop(columns=["interaction"])

# Rename and filter scores
liana_df = liana_df.rename(
    columns={
        "corrected_score_x": "specificity_rank",
        "corrected_score_y": "magnitude_rank",
    }
)

# remove duplicates
liana_df = liana_df.drop_duplicates()

In [4]:
# order by specificity_rank liana_df
liana_df = liana_df.sort_values("specificity_rank") 
#only keep where ranks <0.05
liana_df = liana_df[liana_df["specificity_rank"] < 0.05]
liana_df = liana_df[liana_df["magnitude_rank"] < 0.05]

In [5]:
def create_celltype_interaction_matrix(
    liana_df,
    selected_source=None,
    selected_target=None,
    top_n=20,
    color_scheme="Blue"
):
    """
    Create a matrix plot showing top interactions for each source cell type,
    with target cell type initials displayed in brackets after each interaction.
    When more than 4 sources, splits into two row panels.

    Parameters:
    -----------
    liana_df : pandas.DataFrame
        DataFrame containing ligand-receptor interactions
    selected_source : list, optional
        List of source cell types to include
    selected_target : list, optional
        List of target cell types to include
    top_n : int
        Number of top interactions per source cell type (default: 30)
    color_scheme : str
        Color scheme for the plot ('Blue', 'Red', 'Viridis', 'Cividis')
    """
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import plotly.express as px
    import pandas as pd
    import numpy as np
    import math

    # Filter by selected source and target if provided
    if selected_source:
        liana_df = liana_df[liana_df["source"].isin(selected_source)]
    if selected_target:
        liana_df = liana_df[liana_df["target"].isin(selected_target)]

    # Create unique interaction identifier
    liana_df["interaction"] = (
        liana_df["ligand_complex"] + " → " + liana_df["receptor_complex"]
    )

    # Get unique source cell types
    source_cell_types = sorted(liana_df["source"].unique())
    num_sources = len(source_cell_types)
    
    # Create a function to get cell type initials
    def get_initials(cell_type):
        """Extract initials from cell type name"""
        if pd.isna(cell_type) or cell_type == "":
            return "?"
        # Split by common separators and take first letter of each part
        parts = cell_type.replace("_", " ").replace("-", " ").split()
        if len(parts) == 1:
            # Single word - take first 2-3 characters
            return cell_type[:min(2, len(cell_type))].upper()
        else:
            # Multiple words - take first letter of each
            return "".join([part[0].upper() for part in parts if part])

    # Determine if we need to split into two rows
    split_layout = num_sources > 4
    
    if split_layout:
        # Split sources into two groups
        sources_per_row = math.ceil(num_sources / 2)
        sources_row1 = source_cell_types[:sources_per_row]
        sources_row2 = source_cell_types[sources_per_row:]
        
        # Create two matrices
        plot_matrix1 = np.full((top_n, len(sources_row1)), np.nan)
        cell_annotations1 = np.full((top_n, len(sources_row1)), "", dtype=object)
        hover_text1 = np.full((top_n, len(sources_row1)), "", dtype=object)
        
        plot_matrix2 = np.full((top_n, len(sources_row2)), np.nan)
        cell_annotations2 = np.full((top_n, len(sources_row2)), "", dtype=object)
        hover_text2 = np.full((top_n, len(sources_row2)), "", dtype=object)
        
        matrices = [(plot_matrix1, cell_annotations1, hover_text1, sources_row1),
                   (plot_matrix2, cell_annotations2, hover_text2, sources_row2)]
    else:
        # Single matrix for 4 or fewer sources
        plot_matrix = np.full((top_n, num_sources), np.nan)
        cell_annotations = np.full((top_n, num_sources), "", dtype=object)
        hover_text = np.full((top_n, num_sources), "", dtype=object)
        matrices = [(plot_matrix, cell_annotations, hover_text, source_cell_types)]
    
    # Process data for each matrix
    all_values = []  # Collect all values for consistent color scale
    
    for matrix_idx, (plot_mat, cell_ann, hover_txt, sources) in enumerate(matrices):
        for j, source in enumerate(sources):
            source_data = liana_df[liana_df["source"] == source].copy()
            
            # Get top interactions by specificity rank for this source
            source_interactions = (
                source_data.groupby("interaction")["specificity_rank"]
                .min()
                .reset_index()
                .nsmallest(top_n, "specificity_rank")
            )
            
            # Fill this column with its top interactions
            for i, (_, interaction_row) in enumerate(source_interactions.iterrows()):
                if i >= top_n:  # Safety check
                    break
                    
                interaction = interaction_row["interaction"]
                
                # Get all targets for this interaction from this source
                interaction_data = source_data[source_data["interaction"] == interaction].copy()
                interaction_data = interaction_data.sort_values("specificity_rank")
                
                # Create target initials string
                target_initials = ",".join([
                    get_initials(target) for target in interaction_data["target"].values
                ])
                
                # Get the best specificity and magnitude
                best_row = interaction_data.iloc[0]
                
                # Convert ranks to -log10 scale
                spec_rank = best_row["specificity_rank"]
                mag_rank = best_row["magnitude_rank"]
                
                if spec_rank <= 0:
                    spec_rank = 0.1
                if mag_rank <= 0:
                    mag_rank = 0.1
                    
                spec_log = -np.log10(spec_rank)
                mag_log = -np.log10(mag_rank)
                
                # Fill the matrix
                plot_mat[i, j] = spec_log
                all_values.append(spec_log)
                
                # Create display text
                display_interaction = f"{interaction} [{target_initials}]"
                cell_ann[i, j] = display_interaction
                
                # Create hover text
                hover_txt[i, j] = (
                    f"Source: {source}<br>"
                    f"Rank: {i+1}<br>"
                    f"Interaction: {interaction}<br>"
                    f"Targets (by specificity): [{target_initials}]<br>"
                    f"Specificity (-log10): {spec_log:.3f}<br>"
                    f"Magnitude (-log10): {mag_log:.3f}"
                )
    
    # Set color scheme
    if color_scheme == "Red":
        colorscale = [[0, "white"], [1, "red"]]
    elif color_scheme == "Blue":
        colorscale = [[0, "white"], [1, "#4D4DFF"]]
    elif color_scheme == "Viridis":
        colorscale = px.colors.sequential.Viridis
    elif color_scheme == "Cividis":
        colorscale = px.colors.sequential.Cividis
    else:
        colorscale = [[0, "white"], [1, "#4D4DFF"]]


    
    # Create row labels
    row_labels = [f"Rank {i+1}" for i in range(top_n)]
    
    # Calculate consistent color scale bounds
    zmin = min(all_values) if all_values else 0
    zmax = max(all_values) if all_values else 1
    
    if split_layout:
        # Create subplots for two-row layout
        fig = make_subplots(
            rows=2, cols=1,
            subplot_titles=("",""),#f"Sources: {', '.join(sources_row1)}", 
                           #f"Sources: {', '.join(sources_row2)}"),
            vertical_spacing=0.15,
            row_heights=[0.5, 0.5]
        )
        
        # Add first heatmap
        fig.add_trace(
            go.Heatmap(
                z=plot_matrix1,
                x=sources_row1,
                #y=row_labels,
                colorscale=colorscale,
                hoverongaps=False,
                hovertemplate="%{customdata}<extra></extra>",
                customdata=hover_text1,
                text=cell_annotations1,
                texttemplate="%{text}",
                textfont=dict(size=8, color="black"),
                zmin=zmin,
                zmax=zmax,
                showscale=True,
                colorbar=dict(
                    title=dict(
                        text="Specificity<br>(-log10 rank)",
                        side="right",
                        font=dict(size=14),
                    ),
                    thickness=20,
                    len=0.9,
                    tickfont=dict(size=12),
                    y=0.5,
                ),
            ),
            row=1, col=1
        )
        
        # Add second heatmap
        fig.add_trace(
            go.Heatmap(
                z=plot_matrix2,
                x=sources_row2,
                #y=row_labels,
                colorscale=colorscale,
                hoverongaps=False,
                hovertemplate="%{customdata}<extra></extra>",
                customdata=hover_text2,
                text=cell_annotations2,
                texttemplate="%{text}",
                textfont=dict(size=8, color="black"),
                zmin=zmin,
                zmax=zmax,
                showscale=False,  # Only show one colorbar
            ),
            row=2, col=1
        )
        
        # Update layout for two-row
        plot_height = 800  # Taller for two rows
        plot_width = max(1200, sources_per_row * 250)
        
        fig.update_layout(
            title=dict(
                text="",#f"Top {top_n} Ligand-Receptor Interactions by Source Cell Type<br><sub>Target initials shown in brackets after each interaction, ordered by specificity</sub>",
                font=dict(size=18),
                x=0.5
            ),
            width=plot_width,
            height=plot_height,
            showlegend=False,
            plot_bgcolor="white",
        )
        
        # Update axes
        fig.update_xaxes(tickangle=0, tickfont=dict(size=12), row=1, col=1)
        fig.update_xaxes(tickangle=0, tickfont=dict(size=12), row=2, col=1)
        fig.update_yaxes(title_text="Interaction Specificity Rank", tickfont=dict(size=12), row=1, col=1)
        fig.update_yaxes(title_text="Interaction Specificity Rank", tickfont=dict(size=12), row=2, col=1)
        
    else:
        # Single panel layout for 4 or fewer sources
        fig = go.Figure()
        
        fig.add_trace(
            go.Heatmap(
                z=plot_matrix,
                x=source_cell_types,
                #y=row_labels,
                colorscale=colorscale,
                hoverongaps=False,
                hovertemplate="%{customdata}<extra></extra>",
                customdata=hover_text,
                text=cell_annotations,
                texttemplate="%{text}",
                textfont=dict(size=8, color="black"),
                colorbar=dict(
                    title=dict(
                        text="Specificity<br>(-log10 rank)",
                        side="right",
                        font=dict(size=14),
                    ),
                    thickness=20,
                    len=0.75,
                    tickfont=dict(size=12),
                ),
                zmin=zmin,
                zmax=zmax,
            )
        )
        
        plot_height = 1000
        plot_width = max(1200, num_sources * 250)
        
        fig.update_layout(
            title=dict(
                text=f"Top {top_n} Ligand-Receptor Interactions by Source Cell Type<br><sub>Target initials shown in brackets after each interaction, ordered by specificity</sub>",
                font=dict(size=18),
                x=0.5
            ),
            width=plot_width,
            height=plot_height,
            xaxis=dict(
                title="Source Cell Types",
                tickangle=45,
                tickfont=dict(size=12),
                title_font=dict(size=14),
                side="bottom"
            ),
            yaxis=dict(
                title="Interaction Rank",
                tickfont=dict(size=12),
                title_font=dict(size=14),
                automargin=True,
            ),
            margin=dict(l=150, r=100, t=100, b=150),
            plot_bgcolor="white",
        )

    #remove y axis ticks
    fig.update_yaxes(showticklabels=False)

    # Configure download options
    config = {
        "toImageButtonOptions": {
            "format": "svg",
            "filename": f"celltype_interaction_matrix_top{top_n}",
            "height": plot_height,
            "width": plot_width,
            "scale": 2,
        },
        "displayModeBar": True,
        "scrollZoom": True,
        "staticPlot": False,
    }
    
    # Create summary dataframe
    if split_layout:
        # Combine stats from both matrices
        all_matrices = [plot_matrix1, plot_matrix2]
        all_sources = sources_row1 + sources_row2
        matrix_df = pd.DataFrame({
            'sources': all_sources,
            'num_interactions_found': [
                np.sum(~np.isnan(all_matrices[0][:, j])) if j < len(sources_row1)
                else np.sum(~np.isnan(all_matrices[1][:, j - len(sources_row1)]))
                for j in range(num_sources)
            ],
            'panel': ['Top' if s in sources_row1 else 'Bottom' for s in all_sources]
        })
    else:
        matrix_df = pd.DataFrame({
            'sources': source_cell_types,
            'num_interactions_found': [np.sum(~np.isnan(plot_matrix[:, j])) for j in range(num_sources)]
        })
    
    return fig, config, matrix_df

fig, config, matrix_df = create_celltype_interaction_matrix(liana_df, selected_source=['Endothelial_cells', 'Stem_cells', 'Melanotrophs', 'Immune_cells',
       'Lactotrophs', 'Mesenchymal_cells', 'Somatotrophs',
       'Corticotrophs', 'Gonadotrophs', 'Thyrotrophs'], selected_target=['Endothelial_cells', 'Stem_cells', 'Melanotrophs', 'Immune_cells',
       'Lactotrophs', 'Mesenchymal_cells', 'Somatotrophs',
       'Corticotrophs', 'Gonadotrophs', 'Thyrotrophs'])
#show it
fig.show()

#save as svg and png to /Users/k23030440/Library/CloudStorage/OneDrive-King\'sCollegeLondon/PhD/Year_two/Aim 1/
fig.write_image("/Users/k23030440/Library/CloudStorage/OneDrive-King'sCollegeLondon/PhD/Year_two/Aim 1/celltype_interaction_matrix_top30.svg")
#png
fig.write_image("/Users/k23030440/Library/CloudStorage/OneDrive-King'sCollegeLondon/PhD/Year_two/Aim 1/celltype_interaction_matrix_top30.png")